# DocTamper Dataset Exploration - Phase 0

This notebook demonstrates how to use the `DocForge` Phase 0 pipeline to configure paths, load datasets using LMDB, compute statistics, verify database integrity, and visualize overlay highlights for tampered documents.

## 1. Environment Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add src directory to system path
project_root = Path("").resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import DatasetConfig
from src.dataset import DocTamperDataset
from src.utils import list_available_datasets, dataset_statistics, verify_dataset, overlay_mask
from src.statistics import format_statistics_table
from src.validation import save_validation_report
import matplotlib.pyplot as plt
from PIL import Image

print("Imports successful!")

## 2. Configuration & Dataset Listing

In [ ]:
config = DatasetConfig()
print("Dataset Config Roots:")
print(config)

available = list_available_datasets(config)
print(f"\nAvailable subsets: {available}")

## 3. Load Dataset & Query Sizes

In [ ]:
# Load Training Set
train_dataset = DocTamperDataset(config.training_set)
print(f"Loaded dataset from {train_dataset.db_path.name}")
print(f"Number of samples: {len(train_dataset)}")

## 4. Compute Statistics

Compute statistics on a subset (e.g., 200 samples) to profile the dataset size, resolutions, and tampering area distribution quickly.

In [ ]:
stats = dataset_statistics(train_dataset, sample_limit=200)
table = format_statistics_table(stats)
print(table)

## 5. Verify Dataset Integrity

Check for missing keys, corrupted entries, dimension mismatches, and binarity of masks.

In [ ]:
report = verify_dataset(train_dataset, sample_limit=200)
print(f"Verification status for TrainingSet: {'Passed' if report['passed'] else 'Failed'}")

# Save report markdown
save_validation_report(report, config.reports_dir / "notebook_TrainingSet_validation_report.md")

## 6. Sample Loading and Overlay Visualization

Let's load sample index 0 (which has the `0/1` values anomaly normalized to `0/255` on load) and visualize the original document, binary mask, and highlighting overlay side-by-side.

In [ ]:
# Fetch sample 0
sample = train_dataset[0]
img, msk = sample["image"], sample["mask"]

# Create blend overlay
blend = overlay_mask(img, msk, alpha=0.5, color=(255, 0, 0))

# Plot using Matplotlib
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img)
axes[0].set_title("Original Document", fontsize=14, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(msk, cmap="gray")
axes[1].set_title("Tampering Mask", fontsize=14, fontweight="bold")
axes[1].axis("off")

axes[2].imshow(blend)
axes[2].set_title("Overlay Highlight (Red)", fontsize=14, fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()